In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_accounts_all;
SET search_path TO qdp_accounts_all;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata ( 
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.accounts_all',
  '0.1',
  'bank.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_account table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_account CASCADE;

CREATE TABLE hub_account (
    account_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    account_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
    individual_customer_entity_id VARCHAR,
    individual_entity_id VARCHAR,
    organisation_customer_entity_id VARCHAR,
    organisation_entity_id VARCHAR,
    address_entity_id VARCHAR,
    product_entity_id VARCHAR,
		period_start TIMESTAMP,
		period_end TIMESTAMP
);

ALTER TABLE hub_account ADD CONSTRAINT pk_bankaccounts_hubaccount_accountid PRIMARY KEY (account_id);

COMMENT ON COLUMN hub_account.account_id IS 'Account Identity';



-- -----------------------------------------------------------
-- create sat_account table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_account;
CREATE TABLE sat_account (
    sat_account_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    account_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 0,
    individual_customer_entity_id VARCHAR,  	
    individual_customer_id BIGINT DEFAULT 0,  	
    individual_id BIGINT DEFAULT 0,  	
    organisation_customer_entity_id VARCHAR,  	
    organisation_customer_id BIGINT DEFAULT 0,  	
    organisation_id BIGINT DEFAULT 0,  	
    address_entity_id VARCHAR,
    address_id BIGINT DEFAULT 0,  	
    product_entity_id VARCHAR,
    product_id BIGINT DEFAULT 0,  	
    sort_code VARCHAR,
    account_number VARCHAR,
    account_name VARCHAR,
    iban VARCHAR,
    swiftbic VARCHAR,
	  data_source_code VARCHAR,
	  data_source_concept_id BIGINT,
	  data_source_raw_code VARCHAR,
	  data_source_raw_concept_id BIGINT,
	  type_code VARCHAR,
	  type_concept_id BIGINT,
	  type_raw_code VARCHAR,
	  type_raw_concept_id BIGINT,
    balance DECIMAL(10,2),
    overdraft_limit DECIMAL(10,2),
    loan_original_amount DECIMAL(10,2),
    loan_orininal_maturity_date DATE,
    servicer_identification VARCHAR,
    servicer_scheme_name VARCHAR,    
	  currency_code VARCHAR,
	  currency_concept_id BIGINT,
	  currency_raw_code VARCHAR,
	  currency_raw_concept_id BIGINT,
	  branch_code VARCHAR,
	  branch_concept_id BIGINT,
	  branch_raw_code VARCHAR,
	  branch_raw_concept_id BIGINT,
    opened_datetime TIMESTAMP,
    closed_datetime TIMESTAMP,
	  status_code VARCHAR,
	  status_concept_id BIGINT,
	  status_raw_code VARCHAR,
	  status_raw_concept_id BIGINT,
    risk_rating DECIMAL (10,2),
	  risk_rating_code VARCHAR,
	  risk_rating_concept_id BIGINT,
	  risk_rating_raw_code VARCHAR,
	  risk_rating_raw_concept_id BIGINT,
	  country_code VARCHAR,
	  country_concept_id BIGINT,
	  country_raw_code VARCHAR,
	  country_raw_concept_id BIGINT,
    is_account_alarm BOOLEAN,
    is_bank_account BOOLEAN,
    is_business_account BOOLEAN,
    is_customer_acccount BOOLEAN,
    is_counterparty_account BOOLEAN,
    is_frozed BOOLEAN,
    is_closed BOOLEAN,
    is_open BOOLEAN,
    is_excluded_from_monitoring BOOLEAN,
		period_start TIMESTAMP,
		period_end TIMESTAMP
);

ALTER TABLE sat_account ADD CONSTRAINT pk_accounts_sataccount_sat PRIMARY KEY (sat_account_id);
ALTER TABLE sat_account ADD CONSTRAINT fk_accounts_sataccount_hubaccount_accountid FOREIGN KEY (account_id) REFERENCES hub_account(account_id);


COMMENT ON COLUMN sat_account.account_id IS 'Account Identity';
COMMENT ON COLUMN sat_account.sat_account_id IS 'Sat Account Identity';



-- -----------------------------------------------------------
-- create view_individual_accounts view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_individual_accounts AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
--  icust.customer_name,
--  ihn.given1 AS individual_given_name,
--  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end

  FROM qdp_database.qdp_accounts_all.hub_account h
    JOIN qdp_database.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
--    JOIN qdp_database.qdp_individual_customers.sat_individual_customers icust
--      ON s.individual_customer_id = icust.individual_customer_id
--    JOIN qdp_database.qdp_individuals_all.sat_human_name ihn
--      ON icust.individual_id = ihn.individual_id
    JOIN qdp_database.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN qdp_database.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
--  WHERE ihn.is_trusted = true
  ORDER BY s.account_id ASC
;

SELECT * from view_individual_accounts;





-- -----------------------------------------------------------
-- create view_organisation_accounts view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_organisation_accounts AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
--  icust.customer_name,
--  org.organisation_name AS organisation_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_bank_account,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end

  FROM qdp_database.qdp_accounts_all.hub_account h
    JOIN qdp_database.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
--    JOIN qdp_database.qdp_organisation_customers.sat_organisation_customers icust
--      ON s.organisation_customer_id = icust.organisation_customer_id
--    JOIN qdp_database.qdp_organisations_all.sat_organisation org
--      ON icust.organisation_id = org.organisation_id
    JOIN qdp_database.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN qdp_database.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
  ORDER BY s.account_id ASC
;

SELECT * from view_organisation_accounts;


-- -----------------------------------------------------------
-- create view_accounts_with_alarms view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_accounts_with_alarm AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.account_id,
	h.account_entity_id,
--  icust.customer_name,
--  ihn.given1 AS individual_given_name,
--  ihn.family AS individual_family_name,
	h.period_start AS bank_account_opened,
	h.period_end AS bank_account_closed,
  s.sort_code,
  s.account_number,
  s.account_name,
  s.iban,
  s.swiftbic,
  addr.full_address AS bank_account_address,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id,
  s.type_code,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.balance,
  s.overdraft_limit,
  s.loan_original_amount,
  s.loan_orininal_maturity_date,
  s.servicer_identification,
  s.servicer_scheme_name,    
  s.currency_code,
  s.currency_concept_id,
  s.currency_raw_code,
  s.currency_raw_concept_id,
  s.branch_code,
  s.branch_concept_id,
  s.branch_raw_code,
  s.branch_raw_concept_id,
  s.opened_datetime,
  s.closed_datetime,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.risk_rating,
  s.risk_rating_code,
  s.risk_rating_concept_id,
  s.risk_rating_raw_code,
  s.risk_rating_raw_concept_id,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.is_account_alarm,
  s.is_business_account,
  s.is_customer_acccount,
  s.is_counterparty_account,
  s.is_frozed,
  s.is_closed,
  s.is_open,
  s.is_excluded_from_monitoring,
  s.sat_account_id,
	s.load_datetime,
	s.record_source_id,
  s.individual_customer_entity_id,  	
  s.individual_customer_id,
  s.organisation_customer_entity_id,  	
  s.organisation_customer_id,
  s.address_entity_id,
  s.address_id,
  s.product_entity_id,
  s.product_id,
	s.period_start,
	s.period_end

  FROM qdp_database.qdp_accounts_all.hub_account h
    JOIN qdp_database.qdp_accounts_all.sat_account s
      ON h.account_id = s.account_id
--    JOIN qdp_database.qdp_individual_customers.sat_individual_customers icust
--      ON s.individual_customer_id = icust.individual_customer_id
--    JOIN dqdp_database.qdp_individuals_all.sat_human_name ihn
--      ON icust.individual_id = ihn.individual_id
    JOIN qdp_database.qdp_locations_all.sat_address addr
      ON s.address_id = addr.address_id
    JOIN qdp_database.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
  WHERE s.is_account_alarm = true -- AND ihn.is_trusted = true
  ORDER BY h.account_id ASC
;

SELECT * from view_accounts_with_alarm;




